# HW5 Q2: VGG16 Customized Cost Function
Add a regularization term on the first conv layer's weights: `loss = loss1 + gamma * loss2`, with `loss1 = CrossEntropyLoss` and `loss2 = sum |W_conv1|`.

In [1]:
import os
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import pandas as pd

from models import *   # VGG16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 128
normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms.Compose(
    [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor(), normalize]))
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor(), normalize]))
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

CKPT = 'result/vgg16/model_best.pth.tar'      # trained VGG16 from HW3 (no quantization)


def load_vgg16(path=CKPT):
    model = VGG16()
    model.load_state_dict(torch.load(path)['state_dict'])
    return model.to(device)


def evaluate(model):
    model.eval(); correct = 0
    with torch.no_grad():
        for x, y in testloader:
            correct += model(x.to(device)).argmax(1).eq(y.to(device)).sum().item()
    return 100. * correct / len(testset)


def conv1_abs_sum(model):
    return model.features[0].weight.abs().sum().item()     # first conv layer only

## Step 1: Initial version (loss = loss1)
Load the HW3 checkpoint, no retraining.

In [2]:
model = load_vgg16()
print(model.features[0])
acc0, abs0 = evaluate(model), conv1_abs_sum(model)
print('Test accuracy: %.2f%% | conv1 weight abs sum: %.4f' % (acc0, abs0))

Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
Test accuracy: 93.00% | conv1 weight abs sum: 141.3802


## Cost function and fine-tuning
Every run starts from the step 1 checkpoint and uses exactly the same settings (only `gamma` changes).

In [3]:
lr, momentum, weight_decay, epochs = 0.01, 0.9, 0.0, 10


def finetune(gamma, retrain=False):
    """Fine-tune the step 1 model with loss = loss1 + gamma * loss2. using on disk cache"""
    path = 'result/vgg16_gamma_%g.pth' % gamma
    model = load_vgg16()
    if os.path.exists(path) and not retrain:
        model.load_state_dict(torch.load(path)); return model
    torch.manual_seed(0)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    for epoch in range(epochs):
        model.train()
        for x, y in trainloader:
            x, y = x.to(device), y.to(device)
            output = model(x)
            loss1 = criterion(output, y)
            loss2 = model.features[0].weight.abs().sum()      # loss2: first conv layer only
            loss = loss1 + gamma * loss2                      # gamma = 0 -> loss1, gamma = 1 -> loss1 + loss2
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    torch.save(model.state_dict(), path)
    return model

## Step 2: New loss term (gamma = 1)

In [4]:
model1 = finetune(1.0)
acc1, abs1 = evaluate(model1), conv1_abs_sum(model1)
print('gamma = 1: test accuracy %.2f%% | conv1 weight abs sum %.4f' % (acc1, abs1))

gamma = 1: test accuracy 56.94% | conv1 weight abs sum 21.1252


## Step 3: Sweep gamma
Gamma values 0.001, 0.01, 0.1 in addition to 0 (step 1) and 1 (step 2).

In [5]:
rows = [(0.0, acc0, abs0), (1.0, acc1, abs1)]
for g in [0.001, 0.01, 0.1]:
    m = finetune(g)
    rows.append((g, evaluate(m), conv1_abs_sum(m)))
sweep = pd.DataFrame(rows, columns=['gamma', 'test_acc (%)', 'conv1_abs_sum']).sort_values('gamma').reset_index(drop=True)
sweep

,gamma,test_acc (%),conv1_abs_sum
0,0.000,93.00,141.380203
1,0.001,88.59,41.463493
2,0.010,83.63,10.524588
3,0.100,86.08,5.648850
4,1.000,56.94,21.125196


### Optimal gamma
Chosen as the gamma with accuracy above 80% and the smallest conv1 weight abs sum.

In [6]:
ok = sweep[sweep['test_acc (%)'] > 80]
best = ok.loc[ok['conv1_abs_sum'].idxmin()]
best_gamma = best['gamma']
print('Optimal gamma = %g | test accuracy %.2f%% | conv1 weight abs sum %.4f' % (
    best_gamma, best['test_acc (%)'], best['conv1_abs_sum']))
print()
print('Cost function used:  loss = loss1 + gamma * loss2   with gamma = %g' % best_gamma)
print('  loss1 = CrossEntropyLoss(output, target)')
print('  loss2 = model.features[0].weight.abs().sum()   # first conv layer only')

Optimal gamma = 0.1 | test accuracy 86.08% | conv1 weight abs sum 5.6489

Cost function used:  loss = loss1 + gamma * loss2   with gamma = 0.1
  loss1 = CrossEntropyLoss(output, target)
  loss2 = model.features[0].weight.abs().sum()   # first conv layer only


## Summary table

In [7]:
summary = pd.DataFrame([
    ['1. Initial version', 'loss1', acc0, abs0],
    ['2. New loss term (gamma = 1)', 'loss1 + loss2', acc1, abs1],
    ['3. Fine-tuned (optimal gamma)', 'loss1 + gamma x loss2 (gamma = %g)' % best_gamma,
     best['test_acc (%)'], best['conv1_abs_sum']],
], columns=['Case', 'Cost function', 'Test accuracy (%)', 'Conv1 weight abs sum'])
summary

,Case,Cost function,Test accuracy (%),Conv1 weight abs sum
0,1. Initial version,loss1,93.00,141.380203
1,2. New loss term (gamma = 1),loss1 + loss2,56.94,21.125196
2,3. Fine-tuned (optimal gamma),loss1 + gamma x loss2 (gamma = 0.1),86.08,5.648850


## Discussion

The initial version with just the cross entropy loss resulted in the highest accuracy compared to when the conv1 weighted sum was added to our total cost function. This makes sense because when gamma = 1, we are adding a ton of error in addition to the initial cross entropy loss experienced when training the vgg model, which is why the accuracy is so low. However, we did significantly reduce the conv1 layer weighted abs sum too because we included all of loss2 in our formula now. Smaller loss2 means smaller weights, which is cheaper to quantize in hardware. conv1 is followed by Batch Norm, which can undo the scaling such that we can get more accuracy out of the model. 


The most interesting find is that going from y = 0.1 to 1, the abs sum actually rises, so the function is not monotonically decreasing per se. Each step moves every weight by around lr x y, which is large compared to the weights. They tend to overshoot and oscillate instead of settling, which hurts the accuracy and is why we see there's a large sum at y = 1. Once the best gamma option (gamma = 0.1) was found, we had the lowest optimized conv1 weighted sum and got pretty good accuracy out of it too, although nowhere near the initial case where gamma = 0, since the cost function is still adding some loss2 to the total loss. 